# 08 - modelling

two separate modelling pipelines here. the first (pranav's) runs an xgboost ablation study on a 2024-only dataset with noaa weather data - it tries several feature configurations and measures how much each group contributes. the second (v6) runs walk-forward validation across 6 quarterly folds using both xgboost and lightgbm, covering the full 2023-2024 period.

## pipeline pranav - xgboost ablation (2024 noaa dataset)

this uses a different input dataset to the main pipeline - pranav's combined 2024 parquet which already has the raw noaa weather columns embedded. we aggregate it to hourly zone demand, engineer all the features, cluster zones by demand profile, then run the ablation.

In [ ]:
import datetime
import warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA
import xgboost as xgb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

PIPELINE_VERSION = 'pranav_v1'
RANDOM_STATE     = 42
ZONE_CLUSTER_K   = 5

ROOT        = Path('/Users/Xavier/Desktop/university/YEAR4/datascienceNO3')
INPUT_PATH  = ROOT / 'combined_2024_with_features_with_weather.parquet'
ADJ_PATH    = ROOT / 'taxi_zones_adjacency_matrix.csv'
OUTPUT_DIR  = ROOT / 'outputs' / 'modelling_pranav'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

US_HOLIDAYS = {
    '2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
}
HOLIDAY_DATES = pd.to_datetime(list(US_HOLIDAYS)).normalize()

# noaa col names -> clean names
NOAA_COL_MAP = {
    'HourlyDryBulbTemperature':  'temperature',
    'HourlyPrecipitation':       'precipitation',
    'HourlyWindSpeed':           'windspeed',
    'HourlyWetBulbTemperature':  'wet_bulb_temp',
    'HourlyDewPointTemperature': 'dew_point',
}
WEATHER_FEATURES     = list(NOAA_COL_MAP.values())
WEATHER_LAG_STEPS    = [1, 2, 3, 4, 24, 168]
WEATHER_LAG_FEATURES = [f'{w}_lag_{l}' for w in WEATHER_FEATURES for l in WEATHER_LAG_STEPS]

TEMPORAL_FEATURES = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'is_public_holiday']
CYCLICAL_FEATURES = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
LAG_FEATURES      = ['lag_1', 'lag_2', 'lag_24', 'lag_168', 'lag_336']
SPATIAL_FEATURES  = ['spatial_lag_1', 'spatial_lag_24']
ROLLING_FEATURES  = ['rolling_mean_3h', 'rolling_mean_6h', 'rolling_std_6h']
TARGET            = 'total_trips'

## aggregate trips from the pranav dataset

the input is a 7gb parquet with individual trips and embedded noaa weather. we read it in 2m row chunks to avoid oom, aggregate to hourly zone demand per chunk, then combine.

In [ ]:
def _clean_noaa_string_col(series):
    # noaa stores some numeric fields as strings; 'T' = trace precipitation
    s = series.astype(str).str.strip()
    s = s.replace('T', '0.005')
    return pd.to_numeric(s, errors='coerce').astype('float32')


def aggregate_trips():
    print('\naggregating trips to hourly zone demand')
    needed_cols = ['pickup_datetime', 'pu_zone_id'] + list(NOAA_COL_MAP.keys())
    pf = pq.ParquetFile(INPUT_PATH)

    parts      = []
    total_rows = 0
    CHUNK      = 2_000_000
    for i, batch in enumerate(pf.iter_batches(batch_size=CHUNK, columns=needed_cols)):
        chunk = batch.to_pandas()
        total_rows += len(chunk)
        print(f'\r  chunk {i+1}  ({total_rows:,} rows total)...', end=' ', flush=True)

        chunk['pickup_datetime'] = pd.to_datetime(chunk['pickup_datetime'])
        chunk['hour_slot']       = chunk['pickup_datetime'].dt.floor('h')
        chunk['zone_id']         = chunk['pu_zone_id'].astype('Int32')
        chunk = chunk.dropna(subset=['zone_id'])
        chunk['zone_id'] = chunk['zone_id'].astype(int)

        for noaa_col in ['HourlyPrecipitation', 'HourlyWindSpeed']:
            if noaa_col in chunk.columns:
                chunk[noaa_col] = _clean_noaa_string_col(chunk[noaa_col])
            else:
                chunk[noaa_col] = np.nan

        weather_agg = {nc: 'mean' for nc in NOAA_COL_MAP if nc in chunk.columns}
        agg_spec    = {'pu_zone_id': 'count', **weather_agg}
        grp = (chunk.groupby(['hour_slot', 'zone_id'], sort=False)
                    .agg(agg_spec)
                    .reset_index()
                    .rename(columns={'pu_zone_id': TARGET}))
        parts.append(grp)
        del chunk

    print()
    combined = (
        pd.concat(parts, ignore_index=True)
          .groupby(['hour_slot', 'zone_id'], as_index=False)
          .agg({TARGET: 'sum',
                **{nc: 'mean' for nc in NOAA_COL_MAP if nc in parts[0].columns}})
    )
    print(f'  combined: {len(combined):,} rows, {combined["zone_id"].nunique()} zones')
    return combined


def zero_fill(trips):
    print('  zero-filling grid...', end=' ', flush=True)
    min_h     = trips['hour_slot'].min().floor('D')
    max_h     = trips['hour_slot'].max().ceil('D') - pd.Timedelta(hours=1)
    all_hours = pd.date_range(min_h, max_h, freq='h')
    all_zones = sorted(trips['zone_id'].unique())
    idx       = pd.MultiIndex.from_product([all_hours, all_zones], names=['hour_slot', 'zone_id'])

    weather_cols  = list(NOAA_COL_MAP.keys())
    trips_indexed = trips.set_index(['hour_slot', 'zone_id'])
    full          = trips_indexed.reindex(idx)
    full[TARGET]  = full[TARGET].fillna(0)

    # forward fill weather within zone
    full[weather_cols] = (
        full[weather_cols]
        .groupby(level='zone_id')
        .transform(lambda x: x.ffill().bfill())
    )
    full = full.reset_index()
    print(f'{len(full):,} rows')
    return full


trips = aggregate_trips()
trips = zero_fill(trips)

## feature engineering

same feature set as the main pipeline but using noaa weather columns instead of open-meteo. temporal + cyclical + demand lags + weather lags + rolling + spatial.

In [ ]:
def build_features(df):
    print('\nrenaming weather columns and engineering features')
    df = df.rename(columns=NOAA_COL_MAP)

    for col in WEATHER_FEATURES:
        df[col] = df[col].astype('float32') if col in df.columns else np.float32(0)

    df['hour']              = df['hour_slot'].dt.hour.astype('int8')
    df['day_of_week']       = df['hour_slot'].dt.dayofweek.astype('int8')
    df['month']             = df['hour_slot'].dt.month.astype('int8')
    df['is_weekend']        = df['day_of_week'].isin([5, 6]).astype('int8')
    df['is_rush_hour']      = (df['hour'].between(6, 9) | df['hour'].between(16, 19)).astype('int8')
    df['is_public_holiday'] = df['hour_slot'].dt.normalize().isin(HOLIDAY_DATES).astype('int8')

    h_rad   = 2 * np.pi * df['hour']        / 24
    dow_rad = 2 * np.pi * df['day_of_week'] / 7
    mon_rad = 2 * np.pi * df['month']       / 12
    df['hour_sin']  = np.sin(h_rad).astype('float32')
    df['hour_cos']  = np.cos(h_rad).astype('float32')
    df['dow_sin']   = np.sin(dow_rad).astype('float32')
    df['dow_cos']   = np.cos(dow_rad).astype('float32')
    df['month_sin'] = np.sin(mon_rad).astype('float32')
    df['month_cos'] = np.cos(mon_rad).astype('float32')

    print(f'  zones: {df["zone_id"].nunique()}  |  '
          f'date range: {df["hour_slot"].min().date()} to {df["hour_slot"].max().date()}')
    return df.sort_values(['zone_id', 'hour_slot']).reset_index(drop=True)


def add_lags(df):
    print('\ndemand lag features')
    for lag_h, col in [(1,'lag_1'),(2,'lag_2'),(24,'lag_24'),(168,'lag_168'),(336,'lag_336')]:
        df[col] = df.groupby('zone_id')[TARGET].shift(lag_h)
    return df


def add_weather_lags(df):
    print('  weather lag features...', end=' ', flush=True)
    for wfeat in WEATHER_FEATURES:
        for lag_h in WEATHER_LAG_STEPS:
            df[f'{wfeat}_lag_{lag_h}'] = (
                df.groupby('zone_id')[wfeat].shift(lag_h).astype('float32')
            )
    print(f'done ({len(WEATHER_LAG_FEATURES)} features)')
    return df


def add_rolling_features(df):
    print('  rolling window features...', end=' ', flush=True)
    grp = df.groupby('zone_id')[TARGET]
    df['rolling_mean_3h'] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).astype('float32')
    df['rolling_mean_6h'] = grp.transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean()).astype('float32')
    df['rolling_std_6h']  = grp.transform(lambda x: x.shift(1).rolling(6, min_periods=2).std()).fillna(0).astype('float32')
    print('done')
    return df


def add_spatial_lags(df):
    print('  spatial lag features...', end=' ', flush=True)
    adj_raw          = pd.read_csv(ADJ_PATH, index_col=0)
    adj_raw.index    = adj_raw.index.astype(int)
    adj_raw.columns  = adj_raw.columns.astype(int)
    neighbours       = {z: list(adj_raw.columns[adj_raw.loc[z] > 0])
                        for z in adj_raw.index if z in adj_raw.index}

    demand_pivot = df.pivot(index='hour_slot', columns='zone_id', values=TARGET)
    lag1_piv     = demand_pivot.shift(1)
    lag24_piv    = demand_pivot.shift(24)

    sl1, sl24 = {}, {}
    for z in demand_pivot.columns:
        if z in neighbours:
            nbrs   = [n for n in neighbours[z] if n in demand_pivot.columns]
            sl1[z]  = lag1_piv[nbrs].mean(axis=1)  if nbrs else np.nan
            sl24[z] = lag24_piv[nbrs].mean(axis=1) if nbrs else np.nan

    def pivot_to_df(d, col):
        return (pd.DataFrame(d).stack().reset_index()
                  .rename(columns={'level_0': 'hour_slot', 'level_1': 'zone_id', 0: col}))

    df = df.merge(pivot_to_df(sl1,  'spatial_lag_1'),  on=['hour_slot','zone_id'], how='left')
    df = df.merge(pivot_to_df(sl24, 'spatial_lag_24'), on=['hour_slot','zone_id'], how='left')

    drop_cols = LAG_FEATURES + SPATIAL_FEATURES + ['rolling_mean_3h', 'rolling_mean_6h']
    before    = len(df)
    df        = df.dropna(subset=drop_cols).reset_index(drop=True)
    print(f'{len(df):,} rows after lag nan drop (removed {before - len(df):,})')
    return df


df = build_features(trips)
df = add_lags(df)
df = add_weather_lags(df)
df = add_rolling_features(df)
df = add_spatial_lags(df)

## zone demand profile clustering

we cluster zones into 5 groups based on their mean hourly demand profile (24 hours x 7 days). this gives a zone_cluster feature that lets the model learn different intercepts per cluster without needing a full zone_id embedding.

In [ ]:
def cluster_zones(df):
    print('\nzone demand profile clustering (k=5)')
    profile = (
        df.groupby(['zone_id', 'hour', 'day_of_week'])[TARGET]
          .mean()
          .unstack(['hour', 'day_of_week'])
          .fillna(0)
    )
    X_prof        = StandardScaler().fit_transform(profile.values)
    km            = KMeans(n_clusters=ZONE_CLUSTER_K, random_state=RANDOM_STATE, n_init=20)
    zone_labels   = km.fit_predict(X_prof)
    zone_cluster_map = dict(zip(profile.index, zone_labels))
    df = df.copy()
    df['zone_cluster'] = df['zone_id'].map(zone_cluster_map).fillna(0).astype('int8')

    for c in range(ZONE_CLUSTER_K):
        zones_in_c = [z for z, l in zone_cluster_map.items() if l == c]
        mean_d     = df[df['zone_id'].isin(zones_in_c)][TARGET].mean()
        print(f'  cluster {c}: {len(zones_in_c):3d} zones - mean demand {mean_d:.1f} trips/hr')
    return df


df = cluster_zones(df)
print(f'\n  final dataset: {len(df):,} rows, {df["zone_id"].nunique()} zones')

## xgboost ablation

we define 6 feature variants plus a historical mean baseline, train each on jan-aug 2024 (with sep-oct as early stopping val), and evaluate on nov-dec 2024 test. the variants are: full feature set, no weather, no spatial lags, zone clusters instead of zone_id, weather lags added, and weather lags only.

In [ ]:
def train_models(df):
    print('\nxgboost ablation')

    # 2024 split: jan-aug train | sep-oct val | nov-dec test
    train = df[df['hour_slot'].dt.month.isin(range(1, 9))]
    val   = df[df['hour_slot'].dt.month.isin([9, 10])]
    test  = df[df['hour_slot'].dt.month.isin([11, 12])]
    print(f'  train: {len(train):,}  val: {len(val):,}  test: {len(test):,}')

    base       = (TEMPORAL_FEATURES + CYCLICAL_FEATURES + LAG_FEATURES
                  + SPATIAL_FEATURES + ROLLING_FEATURES + ['zone_id'])
    wlag_feats = [f for f in WEATHER_LAG_FEATURES if f in df.columns]

    xgb_variants = {
        'A_full':              base + WEATHER_FEATURES,
        'B_no_weather':        base,
        'C_no_spatial_lags':   (TEMPORAL_FEATURES + CYCLICAL_FEATURES + LAG_FEATURES
                                 + ROLLING_FEATURES + ['zone_id'] + WEATHER_FEATURES),
        'E_zone_clusters':     (TEMPORAL_FEATURES + CYCLICAL_FEATURES + LAG_FEATURES
                                 + SPATIAL_FEATURES + ROLLING_FEATURES
                                 + ['zone_cluster'] + WEATHER_FEATURES),
        'F_weather_lags_full': base + WEATHER_FEATURES + wlag_feats,
        'G_weather_lags_only': base + wlag_feats,
    }
    for wfeat in WEATHER_FEATURES:
        xgb_variants[f'no_{wfeat}'] = [f for f in base + WEATHER_FEATURES if f != wfeat]

    xgb_params = dict(
        n_estimators=800, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0, objective='reg:squarederror',
        tree_method='hist', random_state=RANDOM_STATE, n_jobs=1, nthread=1,
    )

    def evaluate(y_te, y_pred):
        mae    = mean_absolute_error(y_te, y_pred)
        rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
        mask   = y_te > 0
        mape   = np.mean(np.abs((y_te[mask] - y_pred[mask]) / y_te[mask])) * 100 if mask.any() else np.nan
        ss_res = np.sum((y_te - y_pred) ** 2)
        ss_tot = np.sum((y_te - y_te.mean()) ** 2)
        r2     = 1 - ss_res / ss_tot
        return mae, rmse, mape, r2

    # historical mean baseline
    print('\n  historical mean baseline...', end=' ', flush=True)
    bl_lookup = train.groupby(['zone_id', 'hour', 'day_of_week'])[TARGET].mean()
    zone_mean = train.groupby('zone_id')[TARGET].mean()
    test_bl   = test.join(bl_lookup.rename('bl_pred'), on=['zone_id', 'hour', 'day_of_week'])
    test_bl['bl_pred'] = test_bl['bl_pred'].fillna(test_bl['zone_id'].map(zone_mean))
    bl_mae, bl_rmse, bl_mape, bl_r2 = evaluate(test_bl[TARGET].values, test_bl['bl_pred'].values)
    print(f'mae={bl_mae:.4f}  rmse={bl_rmse:.4f}  mape={bl_mape:.2f}%  r2={bl_r2:.4f}')

    results = {'BASELINE_hist_mean': {'n_features': 0, 'MAE': bl_mae, 'RMSE': bl_rmse,
                                       'MAPE': bl_mape, 'R2': bl_r2, 'best_iter': 0}}
    models, imp_dfs = {}, {}

    print('\n  --- xgboost ---')
    for name, feats in xgb_variants.items():
        feats_ok = [f for f in feats if f in df.columns]
        if not feats_ok:
            print(f'  xgb {name} - skipped (no valid features)')
            continue

        X_tr = train[feats_ok].values.astype(np.float32)
        y_tr = train[TARGET].values.astype(np.float32)
        X_v  = val[feats_ok].values.astype(np.float32)
        y_v  = val[TARGET].values.astype(np.float32)
        X_te = test[feats_ok].values.astype(np.float32)
        y_te = test[TARGET].values

        print(f'  xgb {name} ({len(feats_ok)} features)...', end=' ', flush=True)
        model = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=30)
        model.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)

        y_pred              = model.predict(X_te)
        mae, rmse, mape, r2 = evaluate(y_te, y_pred)
        print(f'mae={mae:.4f}  rmse={rmse:.4f}  mape={mape:.2f}%  r2={r2:.4f}')

        label          = f'XGB_{name}'
        results[label] = {'n_features': len(feats_ok), 'MAE': mae, 'RMSE': rmse,
                           'MAPE': mape, 'R2': r2, 'best_iter': model.best_iteration}
        models[label]  = model
        imp_dfs[label] = pd.Series(
            model.get_booster().get_score(importance_type='gain'), name='gain'
        ).sort_values(ascending=False)

    res_df = pd.DataFrame(results).T
    print('\n  results summary')
    print(res_df[['n_features', 'MAE', 'RMSE', 'MAPE', 'R2']].round(4).to_string())

    bl_val = results['BASELINE_hist_mean']['MAE']
    print(f'\n  improvement over baseline (mae={bl_val:.4f}):')
    for name in [k for k in results if k != 'BASELINE_hist_mean']:
        pct_improvement = (bl_val - results[name]['MAE']) / bl_val * 100
        print(f'    {name:<40} {pct_improvement:+.2f}%')

    res_df.to_csv(OUTPUT_DIR / 'xgb_results.csv')
    run_ts  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    log_row = res_df[['n_features', 'MAE', 'RMSE', 'MAPE', 'R2']].copy()
    log_row.insert(0, 'run_timestamp',    run_ts)
    log_row.insert(1, 'pipeline_version', PIPELINE_VERSION)
    log_path = OUTPUT_DIR / 'results_history.csv'
    log_row.to_csv(log_path, mode='a', header=not log_path.exists())

    return results, models, imp_dfs


results, models, imp_dfs = train_models(df)

## plot ablation results

feature importance (gain) for the full model, a bar chart comparing all core variants, and a per-variable weather ablation showing the mae delta when each weather variable is removed.

In [ ]:
def plot_results(results, imp_dfs):
    print('\nplotting results')
    res_df = pd.DataFrame(results).T

    xgb_a = 'XGB_A_full'
    if xgb_a in imp_dfs:
        imp    = imp_dfs[xgb_a].head(20)
        colors = []
        for f in imp.index:
            if f in WEATHER_FEATURES:    colors.append('#e74c3c')
            elif f in SPATIAL_FEATURES:  colors.append('#e67e22')
            elif f in LAG_FEATURES:      colors.append('#2ecc71')
            elif f in ROLLING_FEATURES:  colors.append('#8e44ad')
            elif f in CYCLICAL_FEATURES: colors.append('#27ae60')
            else:                        colors.append('steelblue')
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(range(len(imp)), imp.values, color=colors)
        ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index, fontsize=9)
        ax.invert_yaxis(); ax.set_xlabel('XGBoost Gain')
        ax.set_title('Feature Importance (Gain) - XGB_A_full  [Pranav dataset]')
        patches = [mpatches.Patch(color=c, label=l) for c, l in [
            ('#e74c3c','Weather'), ('#e67e22','Spatial lags'),
            ('#2ecc71','Temporal lags'), ('#8e44ad','Rolling'),
            ('#27ae60','Cyclical'), ('steelblue','Temporal/zone')]]
        ax.legend(handles=patches, fontsize=7)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'feature_importance_gain.png', dpi=150)
        plt.show()

    # core ablation bar chart
    core = [v for v in ['BASELINE_hist_mean', 'XGB_A_full', 'XGB_B_no_weather',
                         'XGB_C_no_spatial_lags', 'XGB_E_zone_clusters',
                         'XGB_F_weather_lags_full', 'XGB_G_weather_lags_only']
            if v in res_df.index]
    metrics = ['MAE', 'RMSE']
    x       = np.arange(len(metrics))
    width   = 0.8 / len(core)
    palette = plt.cm.tab10.colors

    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (var, color) in enumerate(zip(core, palette)):
        ax.bar(x + i * width, [res_df.loc[var, m] for m in metrics], width, label=var, color=color)
    ax.set_xticks(x + width * (len(core) - 1) / 2)
    ax.set_xticklabels(metrics)
    ax.set_ylabel('Error')
    ax.set_title('Core Variants - MAE and RMSE (Test: Nov-Dec 2024)')
    ax.legend(fontsize=7, loc='upper right')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=150)
    plt.show()

    # per-variable weather ablation
    ablation_keys = [f'XGB_no_{wf}' for wf in WEATHER_FEATURES if f'XGB_no_{wf}' in res_df.index]
    if ablation_keys and xgb_a in res_df.index:
        full_mae = res_df.loc[xgb_a, 'MAE']
        labels   = [k.replace('XGB_no_', '') for k in ablation_keys]
        deltas   = [res_df.loc[k, 'MAE'] - full_mae for k in ablation_keys]
        fig, ax  = plt.subplots(figsize=(8, 4))
        ax.bar(labels, deltas, color=['#e74c3c' if d > 0 else '#2ecc71' for d in deltas])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_ylabel('Delta MAE vs A_full')
        ax.set_title('Per-Variable Weather Ablation')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'weather_ablation_per_var.png', dpi=150)
        plt.show()

    print(f'  saved plots to {OUTPUT_DIR}')


plot_results(results, imp_dfs)

---

## pipeline v6 - walk-forward validation (2023-2024, yellow + green + fhvhv)

this is the main evaluation pipeline. we use all three cab types, cover 2023-2024, and run expanding-window walk-forward validation across 6 quarterly folds. each fold trains on everything before the eval quarter and tests on that quarter. we run xgboost and lightgbm variants.

In [ ]:
import duckdb
import lightgbm as lgb
from sklearn.decomposition import PCA

PIPELINE_VERSION_V6 = 'v6'
PCA_SAMPLE          = 300_000

DATA_DIR_V6  = ROOT / 'data'
WEATHER_PATH = ROOT / 'weather' / 'weather_by_zone_2023_2024_clean.parquet'
OUTPUT_DIR_V6 = ROOT / 'outputs' / 'modelling_v6'
OUTPUT_DIR_V6.mkdir(parents=True, exist_ok=True)

CAB_FILES = {
    'yellow': DATA_DIR_V6 / 'clean_yellow.parquet',
    'green':  DATA_DIR_V6 / 'clean_green.parquet',
    'fhvhv':  DATA_DIR_V6 / 'clean_fhvhv.parquet',
}

WEATHER_FEATURES_V6  = ['temperature_2m', 'precipitation', 'snowfall', 'windspeed_10m', 'weathercode']
TEMPORAL_FEATURES_V6 = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'is_public_holiday']
CYCLICAL_FEATURES_V6 = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
LAG_FEATURES_V6      = ['lag_1', 'lag_2', 'lag_24', 'lag_168', 'lag_336']
SPATIAL_FEATURES_V6  = ['spatial_lag_1', 'spatial_lag_24']
ROLLING_FEATURES_V6  = ['rolling_mean_3h', 'rolling_mean_6h', 'rolling_std_6h']
TARGET_V6            = 'total_trips'

US_HOLIDAYS_V6 = {
    '2023-01-02','2023-01-16','2023-02-20','2023-05-29','2023-06-19',
    '2023-07-04','2023-09-04','2023-10-09','2023-11-10','2023-11-23','2023-12-25',
    '2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
}
HOLIDAY_DATES_V6 = pd.to_datetime(list(US_HOLIDAYS_V6)).normalize()

# expanding window, quarterly evaluation periods
WALK_FORWARD_FOLDS = [
    ('2023-07-01', '2023-07-01', '2023-10-01'),
    ('2023-10-01', '2023-10-01', '2024-01-01'),
    ('2024-01-01', '2024-01-01', '2024-04-01'),
    ('2024-04-01', '2024-04-01', '2024-07-01'),
    ('2024-07-01', '2024-07-01', '2024-10-01'),
    ('2024-10-01', '2024-10-01', '2025-01-01'),
]
ES_VAL_WEEKS = 4

## aggregate all three cab types

we use duckdb here to keep memory usage down - no need to load all three parquets into pandas at once.

In [ ]:
def aggregate_trips_v6():
    print('\naggregating trips to combined hourly zone demand')
    parts = []
    for cab, path in CAB_FILES.items():
        print(f'  {cab}...', end=' ', flush=True)
        sql = f"""
            SELECT
                date_trunc('hour', pickup_datetime)::TIMESTAMP AS hour_slot,
                CAST(pu_zone_id AS INTEGER)                    AS zone_id,
                COUNT(*)                                       AS trip_count
            FROM read_parquet('{path}')
            WHERE pickup_datetime >= '2023-01-01'
              AND pickup_datetime <  '2025-01-01'
              AND pu_zone_id IS NOT NULL
            GROUP BY 1, 2
        """
        df_cab = duckdb.execute(sql).fetchdf()
        print(f'{len(df_cab):,} rows')
        parts.append(df_cab)

    combined = (
        pd.concat(parts, ignore_index=True)
          .groupby(['hour_slot', 'zone_id'], as_index=False)['trip_count']
          .sum()
          .rename(columns={'trip_count': TARGET_V6})
    )
    print(f'  combined: {len(combined):,} rows')
    return combined


def zero_fill_v6(trips):
    print('  zero-filling grid...', end=' ', flush=True)
    all_hours = pd.date_range('2023-01-01', '2024-12-31 23:00', freq='h')
    all_zones = sorted(trips['zone_id'].unique())
    idx = pd.MultiIndex.from_product([all_hours, all_zones], names=['hour_slot', 'zone_id'])
    full = (trips.set_index(['hour_slot', 'zone_id'])
                 .reindex(idx, fill_value=0)
                 .reset_index())
    print(f'{len(full):,} rows')
    return full


trips_v6 = aggregate_trips_v6()
trips_v6 = zero_fill_v6(trips_v6)

## join weather and build features

same feature engineering as the main pipeline - temporal, cyclical, demand lags, rolling, spatial. weather comes from the open-meteo zone parquet (not noaa).

In [ ]:
def join_weather_v6(df):
    print('\njoining weather and engineering features')
    weather = pq.read_table(WEATHER_PATH).to_pandas()
    weather['time'] = pd.to_datetime(weather['time'])
    weather = weather.rename(columns={'time': 'hour_slot', 'LocationID': 'zone_id'})
    weather = weather[['hour_slot', 'zone_id'] + WEATHER_FEATURES_V6]
    df = df.merge(weather, on=['hour_slot', 'zone_id'], how='inner')
    print(f'  after weather join: {len(df):,} rows, {df["zone_id"].nunique()} zones')

    df['hour']              = df['hour_slot'].dt.hour.astype('int8')
    df['day_of_week']       = df['hour_slot'].dt.dayofweek.astype('int8')
    df['month']             = df['hour_slot'].dt.month.astype('int8')
    df['is_weekend']        = df['day_of_week'].isin([5, 6]).astype('int8')
    df['is_rush_hour']      = (df['hour'].between(6, 9) | df['hour'].between(16, 19)).astype('int8')
    df['is_public_holiday'] = df['hour_slot'].dt.normalize().isin(HOLIDAY_DATES_V6).astype('int8')

    h_rad   = 2 * np.pi * df['hour']        / 24
    dow_rad = 2 * np.pi * df['day_of_week'] / 7
    mon_rad = 2 * np.pi * df['month']       / 12
    df['hour_sin']  = np.sin(h_rad).astype('float32')
    df['hour_cos']  = np.cos(h_rad).astype('float32')
    df['dow_sin']   = np.sin(dow_rad).astype('float32')
    df['dow_cos']   = np.cos(dow_rad).astype('float32')
    df['month_sin'] = np.sin(mon_rad).astype('float32')
    df['month_cos'] = np.cos(mon_rad).astype('float32')

    return df.sort_values(['zone_id', 'hour_slot']).reset_index(drop=True)


def add_lags_v6(df):
    print('  temporal lag features...', end=' ', flush=True)
    for lag_h, col in [(1,'lag_1'),(2,'lag_2'),(24,'lag_24'),(168,'lag_168'),(336,'lag_336')]:
        df[col] = df.groupby('zone_id')[TARGET_V6].shift(lag_h)
    print('done')
    return df


def add_rolling_v6(df):
    print('  rolling window features...', end=' ', flush=True)
    grp = df.groupby('zone_id')[TARGET_V6]
    df['rolling_mean_3h'] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).astype('float32')
    df['rolling_mean_6h'] = grp.transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean()).astype('float32')
    df['rolling_std_6h']  = grp.transform(lambda x: x.shift(1).rolling(6, min_periods=2).std()).fillna(0).astype('float32')
    print('done')
    return df


def add_spatial_v6(df):
    print('  spatial lag features...', end=' ', flush=True)
    adj_raw          = pd.read_csv(ADJ_PATH, index_col=0)
    adj_raw.index    = adj_raw.index.astype(int)
    adj_raw.columns  = adj_raw.columns.astype(int)
    neighbours       = {z: list(adj_raw.columns[adj_raw.loc[z] > 0]) for z in adj_raw.index}

    demand_pivot = df.pivot(index='hour_slot', columns='zone_id', values=TARGET_V6)
    lag1_piv     = demand_pivot.shift(1)
    lag24_piv    = demand_pivot.shift(24)

    sl1, sl24 = {}, {}
    for z in demand_pivot.columns:
        if z in neighbours:
            nbrs   = [n for n in neighbours[z] if n in demand_pivot.columns]
            sl1[z]  = lag1_piv[nbrs].mean(axis=1)  if nbrs else np.nan
            sl24[z] = lag24_piv[nbrs].mean(axis=1) if nbrs else np.nan

    def pivot_to_df(d, col):
        return (pd.DataFrame(d).stack().reset_index()
                  .rename(columns={'level_0':'hour_slot','level_1':'zone_id',0:col}))

    df = df.merge(pivot_to_df(sl1,  'spatial_lag_1'),  on=['hour_slot','zone_id'], how='left')
    df = df.merge(pivot_to_df(sl24, 'spatial_lag_24'), on=['hour_slot','zone_id'], how='left')
    before    = len(df)
    drop_cols = LAG_FEATURES_V6 + SPATIAL_FEATURES_V6 + ['rolling_mean_3h', 'rolling_mean_6h']
    df        = df.dropna(subset=drop_cols).reset_index(drop=True)
    print(f'{len(df):,} rows after lag nan drop (removed {before-len(df):,})')
    return df


df_v6 = join_weather_v6(trips_v6)
df_v6 = add_lags_v6(df_v6)
df_v6 = add_rolling_v6(df_v6)
df_v6 = add_spatial_v6(df_v6)
print(f'\n  dataset: {len(df_v6):,} rows, {df_v6["zone_id"].nunique()} zones')

## pca analysis

we fit pca on a sample from the first fold's training data to see how many components explain 90% and 95% of variance. this is analysis only - we don't actually reduce the feature space for modelling.

In [ ]:
def run_pca(df):
    print('\npca (analysis only)')
    features = [f for f in (TEMPORAL_FEATURES_V6 + CYCLICAL_FEATURES_V6 + WEATHER_FEATURES_V6
                              + LAG_FEATURES_V6 + SPATIAL_FEATURES_V6 + ROLLING_FEATURES_V6)
                if f in df.columns]

    fold1_train = df[df['hour_slot'] < pd.Timestamp('2023-07-01')]
    sample      = fold1_train.sample(min(PCA_SAMPLE, len(fold1_train)), random_state=RANDOM_STATE)
    X           = StandardScaler().fit_transform(sample[features].values)

    pca_full   = PCA(random_state=RANDOM_STATE).fit(X)
    evr        = pca_full.explained_variance_ratio_
    cumulative = evr.cumsum()
    n_90 = int(np.argmax(cumulative >= 0.90)) + 1
    n_95 = int(np.argmax(cumulative >= 0.95)) + 1
    print(f'  components for 90% variance: {n_90}  |  95%: {n_95}')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.bar(range(1, min(30, len(evr))+1), evr[:30], color='steelblue')
    ax1.set_xlabel('Principal Component'); ax1.set_ylabel('Explained Variance Ratio')
    ax1.set_title('Scree Plot (first 30 components)')
    ax2.plot(range(1, len(cumulative)+1), cumulative, color='darkorange')
    ax2.axhline(0.90, color='red',   linestyle='--', label=f'90% ({n_90} PCs)')
    ax2.axhline(0.95, color='green', linestyle='--', label=f'95% ({n_95} PCs)')
    ax2.set_xlabel('Number of Components'); ax2.set_ylabel('Cumulative Explained Variance')
    ax2.set_title('Cumulative Explained Variance'); ax2.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR_V6 / 'v6_pca_scree.png', dpi=150)
    plt.show()


run_pca(df_v6)

## walk-forward validation

6 quarterly folds, each expanding the training window. for each fold we train xgboost (3 feature variants) and lightgbm (2 variants) plus a historical mean baseline. early stopping uses the last 4 weeks of each training window, not the eval fold.

In [ ]:
def evaluate_v6(y_te, y_pred):
    mae    = mean_absolute_error(y_te, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
    mask   = y_te > 0
    mape   = np.mean(np.abs((y_te[mask] - y_pred[mask]) / y_te[mask])) * 100
    ss_res = np.sum((y_te - y_pred) ** 2)
    ss_tot = np.sum((y_te - y_te.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}


def train_models_walkforward(df):
    print(f'\nwalk-forward validation - {len(WALK_FORWARD_FOLDS)} folds, quarterly eval windows')

    base = (TEMPORAL_FEATURES_V6 + CYCLICAL_FEATURES_V6 + LAG_FEATURES_V6
            + SPATIAL_FEATURES_V6 + ROLLING_FEATURES_V6 + ['zone_id'])

    xgb_variants = {
        'A_full':            [f for f in base + WEATHER_FEATURES_V6 if f in df.columns],
        'B_no_weather':      [f for f in base if f in df.columns],
        'C_no_spatial_lags': [f for f in (TEMPORAL_FEATURES_V6 + CYCLICAL_FEATURES_V6 + LAG_FEATURES_V6
                                           + ROLLING_FEATURES_V6 + ['zone_id'] + WEATHER_FEATURES_V6)
                              if f in df.columns],
    }
    lgb_variants = {
        'LGBM_A_full':       xgb_variants['A_full'],
        'LGBM_B_no_weather': xgb_variants['B_no_weather'],
    }

    xgb_params = dict(
        n_estimators=800, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0, objective='reg:squarederror',
        tree_method='hist', random_state=RANDOM_STATE, n_jobs=1, nthread=1,
    )
    lgb_params = dict(
        n_estimators=800, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0, objective='regression',
        random_state=RANDOM_STATE, n_jobs=4, verbose=-1,
    )

    all_variant_names = ['BASELINE'] + list(xgb_variants.keys()) + list(lgb_variants.keys())
    fold_records      = {name: [] for name in all_variant_names}

    df = df.copy()
    df['hour_slot'] = pd.to_datetime(df['hour_slot'])

    for fold_i, (train_end_str, val_start_str, val_end_str) in enumerate(WALK_FORWARD_FOLDS, 1):
        train_end = pd.Timestamp(train_end_str)
        val_start = pd.Timestamp(val_start_str)
        val_end   = pd.Timestamp(val_end_str)

        train_full = df[df['hour_slot'] < train_end]
        val_df     = df[(df['hour_slot'] >= val_start) & (df['hour_slot'] < val_end)]

        # last n weeks of train used for early stopping - not the eval fold
        es_cutoff = train_end - pd.Timedelta(weeks=ES_VAL_WEEKS)
        es_val    = train_full[train_full['hour_slot'] >= es_cutoff]
        train_df  = train_full[train_full['hour_slot'] <  es_cutoff]

        quarter_label = f"Q{(val_start.month-1)//3+1}'{str(val_start.year)[2:]}"
        print(f'\n  fold {fold_i}/6  ({quarter_label})')
        print(f'    train : {train_df["hour_slot"].min().date()} to {train_df["hour_slot"].max().date()}  ({len(train_df):,} rows)')
        print(f'    es val: {es_val["hour_slot"].min().date()} to {es_val["hour_slot"].max().date()}  ({len(es_val):,} rows)')
        print(f'    eval  : {val_df["hour_slot"].min().date()} to {val_df["hour_slot"].max().date()}  ({len(val_df):,} rows)')

        bl_lookup = train_full.groupby(['zone_id','hour','day_of_week'])[TARGET_V6].mean()
        zone_mean = train_full.groupby('zone_id')[TARGET_V6].mean()
        val_bl    = val_df.join(bl_lookup.rename('bl_pred'), on=['zone_id','hour','day_of_week']).copy()
        val_bl['bl_pred'] = val_bl['bl_pred'].fillna(val_bl['zone_id'].map(zone_mean))
        bl_m = evaluate_v6(val_bl[TARGET_V6].values, val_bl['bl_pred'].values)
        fold_records['BASELINE'].append({**bl_m, 'fold': fold_i, 'quarter': quarter_label})
        print(f'    baseline              mae={bl_m["MAE"]:.4f}  r2={bl_m["R2"]:.4f}')

        for name, feats in xgb_variants.items():
            X_tr = train_df[feats].values.astype(np.float32)
            y_tr = train_df[TARGET_V6].values.astype(np.float32)
            X_es = es_val[feats].values.astype(np.float32)
            y_es = es_val[TARGET_V6].values.astype(np.float32)
            X_te = val_df[feats].values.astype(np.float32)
            y_te = val_df[TARGET_V6].values

            model = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=30)
            model.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
            m = evaluate_v6(y_te, model.predict(X_te))
            fold_records[name].append({**m, 'fold': fold_i, 'quarter': quarter_label,
                                        'best_iter': model.best_iteration})
            print(f'    xgb {name:<22} mae={m["MAE"]:.4f}  r2={m["R2"]:.4f}  iter={model.best_iteration}')

        for name, feats in lgb_variants.items():
            X_tr = train_df[feats].values.astype(np.float32)
            y_tr = train_df[TARGET_V6].values.astype(np.float32)
            X_es = es_val[feats].values.astype(np.float32)
            y_es = es_val[TARGET_V6].values.astype(np.float32)
            X_te = val_df[feats].values.astype(np.float32)
            y_te = val_df[TARGET_V6].values

            lgb_train = lgb.Dataset(X_tr, label=y_tr)
            lgb_es    = lgb.Dataset(X_es, label=y_es, reference=lgb_train)
            cb        = {k: v for k, v in lgb_params.items() if k not in ('n_estimators','verbose')}
            model_lgb = lgb.train(
                {**cb, 'verbosity': -1}, lgb_train,
                num_boost_round=lgb_params['n_estimators'],
                valid_sets=[lgb_es],
                callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)],
            )
            m = evaluate_v6(y_te, model_lgb.predict(X_te))
            fold_records[name].append({**m, 'fold': fold_i, 'quarter': quarter_label,
                                        'best_iter': model_lgb.best_iteration})
            print(f'    {name:<26} mae={m["MAE"]:.4f}  r2={m["R2"]:.4f}  iter={model_lgb.best_iteration}')

    print('\n  walk-forward summary (mean +/- std across 6 folds)')
    summary_rows = []
    for name in all_variant_names:
        folds = fold_records[name]
        row   = {'model': name}
        for metric in ['MAE', 'RMSE', 'MAPE', 'R2']:
            vals               = [f[metric] for f in folds]
            row[metric]        = np.mean(vals)
            row[f'{metric}_std'] = np.std(vals)
            row[f'{metric}_min'] = np.min(vals)
            row[f'{metric}_max'] = np.max(vals)
        summary_rows.append(row)
        print(f'  {name:<28} mae={row["MAE"]:.4f} +/-{row["MAE_std"]:.4f}  r2={row["R2"]:.4f} +/-{row["R2_std"]:.4f}')

    summary_df = pd.DataFrame(summary_rows).set_index('model')
    summary_df.to_csv(OUTPUT_DIR_V6 / 'v6_walkforward_summary.csv')

    all_fold_rows = []
    for name, folds in fold_records.items():
        for f in folds:
            all_fold_rows.append({'model': name, **f})
    pd.DataFrame(all_fold_rows).to_csv(OUTPUT_DIR_V6 / 'v6_walkforward_folds.csv', index=False)

    return fold_records, summary_df


fold_records, summary_df = train_models_walkforward(df_v6)

## walk-forward plots

three plots: per-fold mae lines over the 6 quarters, mean mae bar chart with std error bars, and r2 stability by quarter.

In [ ]:
def plot_walkforward(fold_records, summary_df):
    print('\nplotting walk-forward results')

    quarters = [fold_records[list(fold_records.keys())[0]][i]['quarter']
                for i in range(len(WALK_FORWARD_FOLDS))]

    palette = {
        'BASELINE':           '#95a5a6',
        'A_full':             '#3498db',
        'B_no_weather':       '#2ecc71',
        'C_no_spatial_lags':  '#e67e22',
        'LGBM_A_full':        '#9b59b6',
        'LGBM_B_no_weather':  '#1abc9c',
    }

    # per-fold mae lines
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(quarters))
    for name, color in palette.items():
        if name not in fold_records: continue
        maes = [f['MAE'] for f in fold_records[name]]
        ls   = '--' if name == 'BASELINE' else '-'
        ax.plot(x, maes, marker='o', label=name, color=color, linestyle=ls, linewidth=2.0)
    ax.set_xticks(x); ax.set_xticklabels(quarters)
    ax.set_xlabel('Evaluation Quarter'); ax.set_ylabel('MAE (trips/hr per zone)')
    ax.set_title('Walk-Forward Validation - Per-Fold MAE  (v6, 263 zones)')
    ax.legend(fontsize=9, loc='upper right'); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR_V6 / 'v6_walkforward_mae.png', dpi=150)
    plt.show()

    # mean mae bar chart with std error bars
    names  = [n for n in palette if n in summary_df.index]
    means  = [summary_df.loc[n, 'MAE']     for n in names]
    stds   = [summary_df.loc[n, 'MAE_std'] for n in names]
    colors = [palette[n] for n in names]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('Mean MAE +/- std (trips/hr per zone)')
    ax.set_title('Walk-Forward Mean MAE across 6 Folds  (v6)')
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    for bar, m, s in zip(bars, means, stds):
        bar_top = bar.get_height() + s + 0.1
        ax.text(bar.get_x() + bar.get_width()/2, bar_top,
                f'{m:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR_V6 / 'v6_mean_mae_bars.png', dpi=150)
    plt.show()

    # r2 stability
    fig, ax = plt.subplots(figsize=(12, 4))
    for name, color in palette.items():
        if name not in fold_records or name == 'BASELINE': continue
        r2s = [f['R2'] for f in fold_records[name]]
        ax.plot(x, r2s, marker='o', label=name, color=color, linewidth=2)
    ax.set_xticks(x); ax.set_xticklabels(quarters)
    ax.set_xlabel('Evaluation Quarter'); ax.set_ylabel('R2')
    ax.set_title('Walk-Forward R2 Stability by Quarter  (v6)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim(0.93, 1.0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR_V6 / 'v6_walkforward_r2.png', dpi=150)
    plt.show()

    print('  saved: v6_walkforward_mae.png, v6_mean_mae_bars.png, v6_walkforward_r2.png')


plot_walkforward(fold_records, summary_df)